# Kanji Scorer Research

In [8]:
import xml.etree.ElementTree as ET
from svgpathtools import parse_path
import numpy as np

def extract_kanjivg_strokes(svg_path, num_samples=30):
    """
    Parses a KanjiVG SVG file and extracts sampled (x, y) coordinates for each stroke.
    """
    tree = ET.parse(svg_path)
    root = tree.getroot()

    # SVG namespaces used by KanjiVG
    ns = {'svg': 'http://www.w3.org/2000/svg'}

    strokes = []

    # Find all path elements (each path corresponds to a stroke in order)
    for path in root.findall('.//svg:path', ns):
        d_attribute = path.get('d')
        if not d_attribute:
            continue

        # Parse Bezier curve into continuous mathematical path
        parsed_path = parse_path(d_attribute)

        # Sample equidistant points along the stroke
        points = []
        for t in np.linspace(0, 1, num_samples):
            point = parsed_path.point(t)
            points.append([point.real, point.imag])

        strokes.append(np.array(points))

    return strokes

# Quick Test:
# Replace with path to any KanjiVG SVG file you downloaded
stroke_data = extract_kanjivg_strokes("data/kanjivg/04e00.svg")
print(f"Total strokes extracted: {len(stroke_data)}")
print(f"First stroke shape: {stroke_data[0].shape}") # Should be (30, 2)

Total strokes extracted: 1
First stroke shape: (30, 2)


In [9]:
import numpy as np

def normalize_character_bbox(strokes, target_size=256.0):
    """
    Rescales and centers all strokes to fit inside a target_size x target_size box.
    """
    # Combine all points across strokes to find global bounding box
    all_points = np.vstack(strokes)
    min_x, min_y = np.min(all_points, axis=0)
    max_x, max_y = np.max(all_points, axis=0)

    width = max_x - min_x
    height = max_y - min_y
    scale = target_size / max(width, height, 1e-5)

    normalized_strokes = []
    for stroke in strokes:
        # Center and scale
        norm_stroke = (stroke - [min_x, min_y]) * scale
        normalized_strokes.append(norm_stroke)

    return normalized_strokes

def resample_stroke_spatial(stroke, num_points=30):
    """
    Resamples a single stroke (N x 2 array) so points are evenly spaced along path length.
    """
    # Calculate cumulative distance along path
    distances = np.sqrt(np.sum(np.diff(stroke, axis=0)**2, axis=1))
    cumulative_dist = np.insert(np.cumsum(distances), 0, 0.0)
    total_length = cumulative_dist[-1]

    if total_length == 0:
        return np.tile(stroke[0], (num_points, 1))

    # Interpolate x and y at uniform distance steps
    uniform_dists = np.linspace(0, total_length, num_points)
    resampled_x = np.interp(uniform_dists, cumulative_dist, stroke[:, 0])
    resampled_y = np.interp(uniform_dists, cumulative_dist, stroke[:, 1])

    return np.column_stack([resampled_x, resampled_y])

# --- Quick Test ---
# Simulated raw stroke input (e.g., fast drawing with 5 points)
raw_user_stroke = np.array([[10, 10], [15, 40], [20, 80], [22, 120], [25, 150]])

# Process
resampled = resample_stroke_spatial(raw_user_stroke, num_points=30)
print(f"Resampled stroke shape: {resampled.shape}") # (30, 2)

Resampled stroke shape: (30, 2)


In [12]:
import numpy as np
from fastdtw import fastdtw
from scipy.spatial.distance import euclidean

def evaluate_stroke_direction(user_stroke, ref_stroke):
    """
    Computes cosine similarity between overall direction vectors of two strokes.
    Returns a score between -1.0 (opposite direction) and 1.0 (identical direction).
    """
    # Vector from start point to end point
    user_vec = user_stroke[-1] - user_stroke[0]
    ref_vec = ref_stroke[-1] - ref_stroke[0]

    norm_user = np.linalg.norm(user_vec)
    norm_ref = np.linalg.norm(ref_vec)

    # Handle zero-length strokes (e.g., dots/taten)
    if norm_user < 1e-5 or norm_ref < 1e-5:
        return 1.0

    cosine_sim = np.dot(user_vec, ref_vec) / (norm_user * norm_ref)
    return float(cosine_sim)

def evaluate_single_stroke(user_stroke, ref_stroke, max_dtw_dist=100.0):
    """
    Evaluates shape match (via FastDTW) and direction match for a single stroke.
    """
    # 1. FastDTW Shape Distance
    dtw_dist, _ = fastdtw(user_stroke, ref_stroke, dist=euclidean)

    # Normalize DTW distance into a 0 - 100 score (lower dist -> higher score)
    shape_score = max(0.0, 100.0 * (1.0 - (dtw_dist / max_dtw_dist)))

    # 2. Direction Check
    dir_sim = evaluate_stroke_direction(user_stroke, ref_stroke)

    # Penalize direction heavily if drawn backwards
    direction_correct = dir_sim > 0.0
    direction_score = max(0.0, dir_sim * 100.0)

    return {
        "dtw_distance": float(dtw_dist),
        "shape_score": round(shape_score, 2),
        "direction_score": round(direction_score, 2),
        "direction_correct": direction_correct
    }

def evaluate_character_sequence(user_strokes, ref_strokes):
    """
    Evaluates sequence order, individual strokes, and overall trajectory score.
    """
    num_user = len(user_strokes)
    num_ref = len(ref_strokes)

    results = []

    # Check total count penalty
    count_match = (num_user == num_ref)

    # Evaluate stroke-by-stroke in drawn sequence
    min_strokes = min(num_user, num_ref)
    for i in range(min_strokes):
        stroke_eval = evaluate_single_stroke(user_strokes[i], ref_strokes[i])
        stroke_eval["stroke_index"] = i + 1
        results.append(stroke_eval)

    avg_shape = np.mean([r["shape_score"] for r in results]) if results else 0.0
    avg_dir = np.mean([r["direction_score"] for r in results]) if results else 0.0

    return {
        "count_match": count_match,
        "user_stroke_count": num_user,
        "target_stroke_count": num_ref,
        "avg_shape_score": round(avg_shape, 2),
        "avg_direction_score": round(avg_dir, 2),
        "stroke_breakdown": results
    }

# --- Quick Test ---
if __name__ == "__main__":
    # Create two synthetic resampled strokes (30 points each)
    ref_stroke = np.column_stack([np.linspace(0, 100, 30), np.zeros(30)]) # Horizontal line L->R

    # Test 1: Similar stroke (drawn left-to-right)
    user_stroke_good = ref_stroke + np.random.normal(0, 1, ref_stroke.shape)

    # Test 2: Reversed stroke (drawn right-to-left)
    user_stroke_backwards = ref_stroke[::-1] + np.random.normal(0, 1, ref_stroke.shape)

    print("Good Stroke Evaluation:", evaluate_single_stroke(user_stroke_good, ref_stroke))
    print("\nBackwards Stroke Evaluation:", evaluate_single_stroke(user_stroke_backwards, ref_stroke))

Good Stroke Evaluation: {'dtw_distance': 41.12844439238093, 'shape_score': 58.87, 'direction_score': 99.99, 'direction_correct': True}

Backwards Stroke Evaluation: {'dtw_distance': 1542.6136156205, 'shape_score': 0.0, 'direction_score': 0.0, 'direction_correct': False}
